# 06-02 学习率调整策略：训练过程中步子要不要变化

前面我们学了优化器：

```text
SGD -> Momentum -> AdaGrad -> RMSProp -> Adam
```

这些优化器主要在回答一个问题：

```text
根据梯度，参数这一步怎么更新？
```

但训练神经网络时，还有一个非常重要的问题：

```text
基础学习率 η 要一直不变吗？
```

这一节就讲学习率调整策略，也叫 learning rate scheduler。

## 1. 学习率到底是什么

先回到最简单的梯度下降公式：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

这里的 $\eta$ 就是学习率。

它决定的是：

```text
沿着梯度方向，一次到底挪多远。
```

所以初学阶段可以把学习率理解成“基础步长”。

学习率太大，可能一步迈过头，损失函数来回震荡。

学习率太小，虽然比较稳，但训练会很慢，甚至长时间看不到明显进步。

一句话记：

```text
学习率控制参数更新的基础步子大小。
```

## 2. 为什么固定学习率不一定合适

训练刚开始时，模型参数通常离好位置很远。

这时候我们希望它走快一点，先大概靠近正确区域。

训练到后面时，模型已经接近比较好的位置。

这时候如果还大步走，就容易在好位置附近来回晃，甚至把已经学好的结果破坏掉。

所以训练过程更像这样：

```text
前期：离目标远，可以大胆走
后期：离目标近，应该小心走
```

这就是学习率调整的基本动机。

不是说固定学习率一定不能用，而是很多时候：

```text
让学习率随着训练过程变化，会更符合训练的节奏。
```

## 3. 学习率调整和优化器不是一回事

这里很容易混。

Adam、RMSProp 也会让不同参数的实际更新步幅不一样，那学习率调整还需要吗？

需要把它们分开看。

优化器更像是在问：

```text
每个参数根据自己的梯度情况，该怎么走？
```

学习率调整更像是在问：

```text
整个训练过程进入不同阶段后，基础步长要不要变？
```

以 Adam 为例，它的简化更新可以写成：

$$
\theta_t=\theta_{t-1}-\eta\frac{m_t}{\sqrt{v_t}+\epsilon}
$$

$m_t$ 和 $v_t$ 是 Adam 根据梯度历史算出来的调整项。

但前面的 $\eta$ 仍然是基础学习率。

所以 scheduler 调的是这个基础学习率 $\eta$，不是直接替代 Adam。

## 4. 第一类：固定学习率

最简单的策略是不调整：

$$
\eta_t=\eta_0
$$

意思是每一轮都用同一个学习率。

这种方法简单、直观、容易调试。

但问题也明显：

```text
前期可能不够快
后期可能不够稳
```

如果你只是做很小的实验，固定学习率完全可以作为起点。

但真正训练较大的神经网络时，通常会考虑让学习率变化。

## 5. 第二类：按阶段下降 Step Decay

Step Decay 的想法非常朴素：

```text
先用一个学习率训练一段时间
到了某些轮数之后，把学习率降下来
```

比如：

```text
第 1 到 30 轮：学习率 0.1
第 31 到 60 轮：学习率 0.01
第 61 轮之后：学习率 0.001
```

可以写成：

$$
\eta_t=\eta_0\gamma^{\left\lfloor\frac{t}{k}\right\rfloor}
$$

这里：

- $\eta_0$：初始学习率
- $\gamma$：每次下降的比例，比如 $0.1$
- $k$：每隔多少轮下降一次
- $t$：当前训练轮数

不要被公式吓到，它只是在表达一句话：

```text
每过 k 轮，学习率乘一次 γ。
```

如果 $\gamma=0.1$，就是每次变成原来的十分之一。

## 6. Step Decay 的优缺点

Step Decay 的优点是非常容易理解。

它符合我们对训练过程的直觉：

```text
先大步找方向
再小步做微调
```

但它也有一个问题：下降发生得比较突然。

比如第 $30$ 轮学习率还是 $0.1$，第 $31$ 轮突然变成 $0.01$。

这种突然变化有时候没问题，有时候会让训练曲线出现明显拐点。

所以 Step Decay 像是手动换挡：

```text
到某个时间点，直接换到更低档。
```

## 7. 第三类：指数衰减 Exponential Decay

指数衰减的想法是：不要突然下降，而是每一轮都稍微降一点。

公式是：

$$
\eta_t=\eta_0\gamma^t
$$

这里 $\gamma$ 通常小于 $1$，比如：

$$
\gamma=0.95
$$

那么学习率会变成：

```text
η0
η0 × 0.95
η0 × 0.95²
η0 × 0.95³
...
```

它的特点是下降更平滑。

一句话记：

```text
Step Decay 是隔一段时间降一次；指数衰减是每次都慢慢降一点。
```

## 8. 指数衰减要注意什么

指数衰减的关键在 $\gamma$。

如果 $\gamma$ 太小，学习率会下降得太快。

比如：

$$
\gamma=0.5
$$

那每一轮学习率都变成原来的一半，很快就小到几乎走不动。

如果 $\gamma$ 太接近 $1$，学习率下降得很慢，后期可能还是不够稳。

所以指数衰减的核心不是记公式，而是理解它的节奏：

```text
γ 越小，降得越快
γ 越接近 1，降得越慢
```

## 9. 第四类：余弦退火 Cosine Annealing

余弦退火听起来很吓人，但直觉很简单：

```text
学习率从大到小，平滑地滑下来。
```

常见形式是：

$$
\eta_t=\eta_{\min}+\frac{1}{2}(\eta_{\max}-\eta_{\min})\left(1+\cos\left(\frac{t}{T}\pi\right)\right)
$$

这里：

- $\eta_{\max}$：最大学习率
- $\eta_{\min}$：最小学习率
- $T$：一个下降周期的长度
- $t$：当前在这个周期里的位置

公式不用死背。你只要知道：

```text
cos 从 1 平滑变到 -1
所以学习率也能从大平滑变到小
```

它不像 Step Decay 那样突然跳变，而是更平滑地减小学习率。

## 10. 什么是退火

退火这个词来自材料加工。

金属加热后慢慢冷却，内部结构会更稳定。

放到神经网络训练里，可以这样理解：

```text
训练前期学习率大，探索范围大
训练后期学习率小，慢慢稳定下来
```

所以“余弦退火”不是说神经网络真的像金属一样，而是借这个词描述：

```text
学习率逐渐冷却，参数更新逐渐稳定。
```

## 11. 第五类：Warmup 预热

前面一直在说学习率后期要变小。

但有些模型在训练刚开始时，也不适合马上用很大的学习率。

为什么？

因为刚开始参数是随机初始化的，梯度也可能不稳定。

如果一上来就大步走，模型可能还没站稳就被推得很乱。

Warmup 的想法是：

```text
刚开始先用小学习率
然后慢慢升到目标学习率
等训练稳定后，再进入正常衰减
```

最常见的是线性 warmup：

$$
\eta_t=\eta_{\max}\frac{t}{T_{warmup}}
$$

这里 $T_{warmup}$ 是预热阶段长度。

当 $t$ 从 $0$ 增加到 $T_{warmup}$ 时，学习率从 $0$ 慢慢增加到 $\eta_{\max}$。

## 12. Warmup 什么时候常见

Warmup 在大模型、Transformer、较深网络里很常见。

原因是这些模型训练初期更容易不稳定。

你可以把 warmup 想成开车起步：

```text
不是一脚油门到底
而是先慢慢提速
```

所以完整训练节奏经常是：

```text
先 warmup：从小学习率升上来
再 decay：训练后期慢慢降下去
```

这就同时照顾了训练初期和训练后期。

## 13. 为什么不是学习率越小越好

学习率小确实更稳，但不代表越小越好。

如果学习率太小，参数每一步变化都很小。

这样会出现两个问题：

```text
训练速度很慢
可能卡在不太好的区域，很久走不出来
```

所以学习率不是单纯追求小，而是追求合适。

更准确地说：

```text
前期需要足够大，帮助模型快速靠近好区域
后期需要足够小，帮助模型在好区域稳定下来
```

## 14. 为什么不是学习率越大越好

学习率大可以让训练前期下降更快。

但如果太大，参数会一步迈过好位置。

用损失函数的角度看，就是：

```text
本来想往谷底走
结果一步跨过谷底
下一步又从另一边跨回来
损失来回震荡
```

严重时，损失甚至可能越来越大。

所以如果训练中看到 loss 大幅波动、不下降、甚至变成异常值，学习率过大是一个常见怀疑对象。

## 15. 常见策略怎么选

初学阶段不用把所有 scheduler 都背下来。

先记这个选择思路：

| 策略 | 直觉 | 适合初学怎么记 |
| --- | --- | --- |
| 固定学习率 | 一直同样步子 | 最简单的基线 |
| Step Decay | 到点突然降档 | 手动换挡 |
| Exponential Decay | 每次都降一点 | 慢慢减速 |
| Cosine Annealing | 平滑滑到较小值 | 平滑冷却 |
| Warmup | 开始先慢慢升 | 先热身再正常跑 |

如果你不知道先用什么，可以这样想：

```text
小实验：固定学习率或 Step Decay
想更平滑：Cosine Annealing
模型很深或训练初期不稳定：加 Warmup
```

## 16. 和我们前面学过内容的关系

现在把训练更新的逻辑串起来：

```text
损失函数：告诉我们现在错得多不多
反向传播：算出每个参数应该怎么改
优化器：根据梯度决定参数更新方式
学习率：控制基础步子大小
学习率调整策略：让基础步子随训练阶段变化
```

所以 scheduler 不是孤立概念。

它是在整个训练流程里控制节奏的东西。

## 17. 本节总结

学习率调整策略的核心逻辑是：

```text
训练前期离目标远，可以走大步
训练后期接近好位置，应该走小步
训练刚开始不稳定时，可以先 warmup
```

先记住一句话：

```text
优化器决定怎么走，学习率决定基础步子多大，scheduler 决定这个步子随时间怎么变。
```

学到这里，神经网络训练的主线已经比较完整了：

```text
前向传播 -> 损失函数 -> 反向传播 -> 优化器 -> 学习率调整
```